# Phase 4 GCC Constraints — Run Record

This notebook loads and displays Phase 4 GCC-constrained fault injection results.
Runs are executed via CLI (`scripts/run_phase4.py`); this notebook is for inspection.

In [1]:
import yaml
import json
import sys
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv(Path("../../.env"))

sys.path.insert(0, str(Path("../../src")))
sys.path.insert(0, str(Path("../../")))

config_path = Path("../../config/phase4.yaml")
raw = yaml.safe_load(config_path.read_text())
print("Config:")
print(yaml.dump(raw, default_flow_style=False))

Config:
artifacts:
  output_dir: artifacts/
fault_types:
- after_seconds: 3.0
  name: timeout
- fail_on_call: 1
  name: malformed_json
- name: checkpoint_loss
- name: context_overflow
  token_limit: 200
gcc:
  egress_jitter_ms: 50
  latency_ms: 200
  tpm_limit: 60000
llm:
  api_key: ${OPENAI_API_KEY}
  base_url: ${OPENAI_BASE_URL}
  max_tokens: 1024
  model: openai/openai/gpt-4o-mini
  request_timeout: 120
  temperature: 0.0
mcp:
  workspace_root: workspace/
run:
  inter_run_delay_s: 5
  n_runs: 5
  run_id_prefix: gcc



In [2]:
# Load existing Phase 4 artifacts
from analysis.aggregate import load_runs
import pandas as pd

df = load_runs("../../artifacts/", prefix="gcc")
print(f"Loaded {len(df)} Phase 4 (GCC) runs")
display(df[["run_id", "failure_type", "recovery_mode", "total_latency_s", "output_quality", "gcc_latency_ms", "gcc_rate_wait_s"]].head(30))

Loaded 20 Phase 4 (GCC) runs


,run_id,failure_type,recovery_mode,total_latency_s,output_quality,gcc_latency_ms,gcc_rate_wait_s
0,gcc_checkpointloss_49c0a441,checkpoint_loss,manual,30.825786,1.0,200,0.0
1,gcc_checkpointloss_6abc6470,checkpoint_loss,manual,32.771477,1.0,200,0.0
2,gcc_checkpointloss_a87ffa01,checkpoint_loss,manual,33.702484,1.0,200,0.0
3,gcc_checkpointloss_bdbf3650,checkpoint_loss,manual,36.410100,1.0,200,0.0
4,gcc_checkpointloss_f2e2d435,checkpoint_loss,manual,35.416325,1.0,200,0.0
5,gcc_contextoverflow_454b1595,context_overflow,manual,0.251462,0.0,200,0.0
6,gcc_contextoverflow_7aaddebd,context_overflow,manual,0.253593,0.0,200,0.0
7,gcc_contextoverflow_a172de1a,context_overflow,manual,0.251585,0.0,200,0.0
8,gcc_contextoverflow_b9dcd3ff,context_overflow,manual,0.218468,0.0,200,0.0
9,gcc_contextoverflow_d431c5d4,context_overflow,manual,0.204050,0.0,200,0.0


In [3]:
# Recovery summary table (GCC-constrained)
if not df.empty:
    summary = df.groupby("failure_type").agg(
        runs=("run_id", "count"),
        automatic=("recovery_mode", lambda x: (x == "automatic").sum()),
        manual=("recovery_mode", lambda x: (x == "manual").sum()),
        unrecoverable=("recovery_mode", lambda x: (x == "unrecoverable").sum()),
        avg_latency_s=("total_latency_s", "mean"),
        avg_output_quality=("output_quality", "mean"),
    ).round(3)
    display(summary)
    print(f"\nTotal: {len(df)} runs across {df['failure_type'].nunique()} fault types")

,runs,automatic,manual,unrecoverable,avg_latency_s,avg_output_quality
failure_type,,,,,,
checkpoint_loss,5,0,5,0,33.825,1.0
context_overflow,5,0,5,0,0.236,0.0
malformed_json,5,5,0,0,0.229,0.0
timeout,5,0,5,0,3.232,0.0



Total: 20 runs across 4 fault types


In [4]:
# Phase 3 vs Phase 4 comparison
p3 = load_runs("../../artifacts/", prefix="failures")
print("=== Phase 3 (no GCC) ===")
if not p3.empty:
    s3 = p3.groupby("failure_type").agg(
        runs=("run_id", "count"),
        avg_latency=("total_latency_s", "mean"),
        recovery=("recovery_mode", lambda x: x.mode().iloc[0] if len(x) > 0 else "n/a"),
    ).round(3)
    display(s3)

print("\n=== Phase 4 (GCC) ===")
if not df.empty:
    s4 = df.groupby("failure_type").agg(
        runs=("run_id", "count"),
        avg_latency=("total_latency_s", "mean"),
        recovery=("recovery_mode", lambda x: x.mode().iloc[0] if len(x) > 0 else "n/a"),
    ).round(3)
    display(s4)

print("\n=== Latency delta (GCC overhead) ===")
if not p3.empty and not df.empty:
    for ft in df["failure_type"].unique():
        p3_lat = p3[p3["failure_type"] == ft]["total_latency_s"].mean()
        p4_lat = df[df["failure_type"] == ft]["total_latency_s"].mean()
        print(f"  {ft}: {p3_lat:.3f}s → {p4_lat:.3f}s  (delta: +{p4_lat - p3_lat:.3f}s)")

=== Phase 3 (no GCC) ===


,runs,avg_latency,recovery
failure_type,,,
checkpoint_loss,5,37.816,manual
context_overflow,5,0.003,manual
malformed_json,5,0.002,automatic
timeout,5,3.004,manual



=== Phase 4 (GCC) ===


,runs,avg_latency,recovery
failure_type,,,
checkpoint_loss,5,33.825,manual
context_overflow,5,0.236,manual
malformed_json,5,0.229,automatic
timeout,5,3.232,manual



=== Latency delta (GCC overhead) ===
  checkpoint_loss: 37.816s → 33.825s  (delta: +-3.991s)
  context_overflow: 0.003s → 0.236s  (delta: +0.232s)
  malformed_json: 0.002s → 0.229s  (delta: +0.226s)
  timeout: 3.004s → 3.232s  (delta: +0.228s)
